In [1]:
# Install the core stack!pip install -

!pip install -q langgraph groq langchain langchain-community sentence-transformers scikit-learn pypdf python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
!pip install -U langchain-text-splitters langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.10
    Uninstalling langgraph-prebuilt-1.0.10:
      Successfully uninstalled langgraph-prebuilt-1.0.10
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.9
    Uninstalling langgraph-1.1.9:
      Successfully uninstalled langgraph-1.1.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.15
    Uninstalling langchain-1.2.15:
      Successfully uninstalled langchain-1.2.15


In [3]:
import os
import getpass
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import numpy as np

# For the Graph logic later
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

# Set up your Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [5]:
!wget -O sample_writing.txt https://www.gutenberg.org/cache/epub/1661/pg1661.txt

--2026-05-05 08:30:41--  https://www.gutenberg.org/cache/epub/1661/pg1661.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607609 (593K) [text/plain]
Saving to: ‘sample_writing.txt’

sample_writing.txt  100%[===================>] 593.37K  3.69MB/s    in 0.2s    

2026-05-05 08:30:41 (3.69 MB/s) - ‘sample_writing.txt’ saved [607609/607609]



In [15]:
class GhostWriterState(TypedDict):
    user_query: str           # The topic you want to write about
    style_samples: List[str]  # The 5 chunks from the Collector
    style_dna: str            # The LLM's analysis of the style
    draft: str                # The generated text
    revision_notes: str       # Any feedback for the generator(red pen)
    iteration_count: int      # To prevent infinite loops

In [11]:
import os
import getpass

# 1. Groq Key (You already have this)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# 2. Hugging Face Token (Add this now)
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Enter your Hugging Face Token: ")
# This line officially logs you into the HF Hub for the current session
from huggingface_hub import login
login(os.environ["HF_TOKEN"])

Enter your Hugging Face Token: ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [20]:
#first node collector
def collector_node(state: GhostWriterState):
    print("--- COLLECTING & FILTERING STYLE SAMPLES ---")

    loader = TextLoader("/content/sample_writing.txt")
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=50)
    raw_chunks = text_splitter.split_documents(docs)

    # --- NEW: Pre-Filtering Step ---
    # We remove chunks that look like legal jargon, headers, or metadata
    forbidden_keywords = ["gutenberg", "license", "copyright", "terms of use", "header", "footer","Gutenberg™"]

    clean_chunk_texts = []
    for c in raw_chunks:
        content_lower = c.page_content.lower()
        # Only keep the chunk if NONE of the forbidden keywords are in it
        if not any(word in content_lower for word in forbidden_keywords):
            clean_chunk_texts.append(c.page_content)

    # --- Proceed with K-Means on the CLEAN chunks ---
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(clean_chunk_texts)

    num_clusters = 5
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    kmeans.fit(embeddings)

    representative_chunks = []
    for i in range(num_clusters):
        distances = np.linalg.norm(embeddings - kmeans.cluster_centers_[i], axis=1)
        closest_index = np.argmin(distances)
        representative_chunks.append(clean_chunk_texts[closest_index])

    # Update the state
    return {"style_samples": representative_chunks}


In [21]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 1.2.0
    Uninstalling groq-1.2.0:
      Successfully uninstalled groq-1.2.0


In [32]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

def analysis_node(state: GhostWriterState):
    print("--- ANALYZING STYLE DNA ---")
    llm = ChatGroq(model="llama-3.1-8b-instant")

    # Combine samples into one string for the LLM
    samples_text = "\n\n".join(state["style_samples"])

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert linguistic profiler. Analyze the following text samples and create a 'Style DNA' profile. "
                   "Focus on: sentence structure, vocabulary level, use of imagery, and tone. "
                   "Be specific. Do not mention the content, only the writing style."),
        ("user", f"Samples:\n{samples_text}")
    ])

    chain = prompt | llm
    response = chain.invoke({})

    return {"style_dna": response.content}

In [33]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

def generation_node(state: GhostWriterState):
    print("--- GENERATING DRAFT ---")
    llm = ChatGroq(model="llama-3.1-8b-instant")

    # We combine the DNA and the User Query into a specialized prompt
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a master ghostwriter. Your goal is to write a response to the user's topic "
                   "using the following 'Style DNA' profile strictly: \n\n {style_dna}"),
        ("user", "Write a short piece (2-3 paragraphs) about the following topic: {user_query}")
    ])

    # If this is a REVISION (from the Auditor), we add the revision notes to the prompt
    if state.get("revision_notes"):
        prompt.append(("assistant", f"Previous Draft: {state['draft']}"))
        prompt.append(("user", f"Please revise the draft based on these notes: {state['revision_notes']}"))

    chain = prompt | llm
    response = chain.invoke({
        "style_dna": state["style_dna"],
        "user_query": state["user_query"]
    })

    # Update the draft and increment iteration
    return {
        "draft": response.content,
        "iteration_count": state.get("iteration_count", 0) + 1
    }

In [34]:
def auditor_node(state: GhostWriterState):
    print("--- AUDITING THE STYLE ---")
    llm = ChatGroq(model="llama-3.3-70b-versatile") # Using a larger model for better "criticism"

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a strict editor. Compare the 'Draft' against the 'Style DNA'. "
                   "If the draft perfectly matches the style, respond with 'MATCH'. "
                   "If it does not match, provide specific instructions on how to fix it."),
        ("user", f"Style DNA: {state['style_dna']}\n\nDraft: {state['draft']}")
    ])

    response = llm.invoke(prompt.format())

    # Logic to decide whether to loop or end
    if "MATCH" in response.content.upper() or state["iteration_count"] >= 3:
        return {"revision_notes": "", "matches": True}
    else:
        return {"revision_notes": response.content, "matches": False}

In [25]:
from langgraph.checkpoint.memory import MemorySaver

# 1. Initialize the memory saver
memory = MemorySaver()

In [35]:
workflow = StateGraph(GhostWriterState)

# Add Nodes
workflow.add_node("collector", collector_node)
workflow.add_node("analyzer", analysis_node)
workflow.add_node("generator", generation_node)
workflow.add_node("auditor", auditor_node)

# Build Edges
workflow.set_entry_point("collector")
workflow.add_edge("collector", "analyzer")
workflow.add_edge("analyzer", "generator")
workflow.add_edge("generator", "auditor")

# The "Decision" Loop
workflow.add_conditional_edges(
    "auditor",
    lambda x: "end" if x["matches"] else "generator",
    {
        "end": END,
        "generator": "generator"
    }
)

app = workflow.compile(checkpointer= memory)

In [36]:
# Define your session configuration
config = {"configurable": {"thread_id": "sherlock_session_1"}}

# First Turn: Provide the file and query
inputs = {
    "user_query": "Write a short note about the invention of the internet.",
    "iteration_count": 0
}

# Stream the results
for event in app.stream(inputs, config=config):
    for node, value in event.items():
        print(f"Node '{node}' finished.")
        if "draft" in value:
            print(f"\nFinal Draft:\n{value['draft']}")

--- COLLECTING & FILTERING STYLE SAMPLES ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Node 'collector' finished.
--- ANALYZING STYLE DNA ---
Node 'analyzer' finished.
--- GENERATING DRAFT ---
Node 'generator' finished.

Final Draft:
The invention of the internet, a pivotal moment in the annals of technological history, was the culmination of a series of skirmishes between visionary researchers and the unyielding obstacles of the early computing era. It was in the late 1960s that the United States Department of Defense's Advanced Research Projects Agency (ARPA) initiated a project, dubbed ARPANET, with the express purpose of creating a network of computers that could communicate with one another in real-time. The problems which were submitted to the project's lead researchers, a coterie of brilliant engineers and computer scientists, were nothing short of daunting: how to design a system that could transmit data between disparate nodes, thereby facilitating the sharing of information across vast distances. The challenge was akin to solving a complex puzzle, with each pie